In [ ]:
import pandas as pd
from string import punctuation
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.metrics import precision_score, recall_score, accuracy_score
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
reviews = pd.read_csv("cw_data.csv",header=None)

In [ ]:
reviews.head()

In [ ]:
print(repr(reviews[0]))

In [ ]:
def lowercase(texts):
  return [t.lower() for t in texts]

In [ ]:
def remove_punctuation(texts):
  cleaned_texts = []
  for t in texts:
    t = t.translate(str.maketrans(punctuation, ' ' * len(punctuation)))
    cleaned_texts.append(t)
  return cleaned_texts

In [ ]:
def remove_hidden_characters(texts):
  cleaned_texts = []
  for t in texts:
    cleaned_texts.append(t.replace("—", " ").replace("…", " ").replace("→"," "))
  return cleaned_texts

In [ ]:
def abbreviation_removal(texts):
  cleaned_texts = []
  for t in texts:
    cleaned_texts.append(t.replace("3rd","third").replace("2fa","2-factor-authentication"))
  return cleaned_texts

In [ ]:
def whitespace_removal(texts):
  return [t.strip() for t in texts]

In [ ]:
def trigram_tokenise(texts):
  trigram_texts = []

  for t in texts:
    trigrams = [t[i:i+3] for i in range(len(t) - 3 + 1)]
    trigram_texts.append(trigrams)

  return trigram_texts

In [ ]:
def remove_numbers(texts):
    return [[word for word in text if not word.isdigit()] for text in texts]

In [ ]:
from collections import Counter

def remove_rare_words(texts):
  tokens = [token for text in texts for token in text]
  frequency = Counter(tokens)
  cleaned_texts = []
  for text in texts:
    cleaned = [token for token in text if 3 <= frequency[token]]
    cleaned_texts.append(cleaned)

  return cleaned_texts

In [ ]:
from nltk.parse.transitionparser import remove
def preprocess(texts):
  texts = lowercase(texts)
  texts = remove_hidden_characters(texts)
  texts = abbreviation_removal(texts)
  texts = remove_punctuation(texts)
  texts = whitespace_removal(texts)
  texts = trigram_tokenise(texts)
  texts = remove_numbers(texts)
  texts = remove_rare_words(texts)
  return texts

In [ ]:
def build_vocabulary(texts):
    vocabulary = {}
    for t in texts:
      for w in t:
        if w not in vocabulary:
          vocabulary[w] = len(vocabulary)
    return vocabulary

In [ ]:
def encode(texts, vocabulary=None):
  vocabulary = build_vocabulary(texts) if vocabulary is None else vocabulary
  dataset = []
  for t in texts:
    feature_vector = [0] * len(vocabulary)
    for w in t:
      if w in vocabulary:
        feature_vector[vocabulary[w]] = 1
    dataset.append(feature_vector)
  return np.array(dataset), vocabulary

In [ ]:
#from sklearn.feature_extraction.text import TfidfTransformer

accuracy_scores = []
precision_scores = []
recall_scores = []
true_labels = []
predicted_labels = []

for i in range(25,35):
  preprocessed = preprocess(reviews[0])
  features, train_vocabulary = encode(preprocessed)

  #tfidf = TfidfTransformer()
  #features = tfidf.fit_transform(features)
  targets = reviews[1]

  train_features, test_features, train_targets, test_targets = train_test_split(
      features, targets, test_size=0.3, random_state=i)
  clf = LogisticRegression(random_state=0).fit(train_features, train_targets)
  y_pred = clf.predict(test_features)
  true_labels.extend(test_targets.array)
  predicted_labels.extend(y_pred)
  accuracy_scores.append(accuracy_score(test_targets, y_pred))
  precision_scores.append(precision_score(test_targets, y_pred, average='macro'))
  recall_scores.append(recall_score(test_targets, y_pred, average='macro'))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
results = ConfusionMatrixDisplay.from_predictions(true_labels, predicted_labels)
plt.xticks(rotation=90)

In [ ]:
print("Accuracy: ", np.mean(accuracy_scores))
print("Precision: ", np.mean(precision_scores))
print("Recall: ", np.mean(recall_scores))